In [0]:
   CREATE CATALOG IF NOT EXISTS ecom_dev
     MANAGED LOCATION 'abfss://lakehouse@stecomdev321.dfs.core.windows.net/ecom_dev'
     COMMENT 'E-commerce lakehouse (dev)';
DESCRIBE CATALOG EXTENDED ecom_dev;
CREATE SCHEMA IF NOT EXISTS ecom_dev.bronze    COMMENT 'Raw, append-only copies of source data plus ingestion metadata';
CREATE SCHEMA IF NOT EXISTS ecom_dev.silver    COMMENT 'Cleaned, typed, deduplicated, validated data';
CREATE SCHEMA IF NOT EXISTS ecom_dev.gold      COMMENT 'Business-ready star schema and aggregates';
CREATE SCHEMA IF NOT EXISTS ecom_dev.reporting COMMENT 'Stable views consumed by Power BI';
CREATE SCHEMA IF NOT EXISTS ecom_dev.control   COMMENT 'Config, watermarks, audit and data quality tables';
CREATE SCHEMA IF NOT EXISTS ecom_dev.ml        COMMENT 'Features, forecasts and model monitoring';
SHOW SCHEMAS IN ecom_dev;

CREATE EXTERNAL VOLUME IF NOT EXISTS ecom_dev.bronze.landing
  LOCATION 'abfss://landing@stecomdev321.dfs.core.windows.net/ecommerce'
  COMMENT 'Raw source files delivered by ADF';

CREATE VOLUME IF NOT EXISTS ecom_dev.bronze.checkpoints
  COMMENT 'Auto Loader checkpoints and inferred schema locations';

  LIST '/Volumes/ecom_dev/bronze/landing/';
LIST '/Volumes/ecom_dev/bronze/landing/orders';


CREATE TABLE IF NOT EXISTS ecom_dev.control.pipeline_config (
  source_table   STRING        NOT NULL,
  source_system  STRING,
  load_type      STRING        NOT NULL,
  partition_key  STRING,
  file_format    STRING        NOT NULL,
  bronze_table   STRING        NOT NULL,
  silver_table   STRING        NOT NULL,
  primary_keys   ARRAY<STRING> NOT NULL,
  order_by_col   STRING,
  scd_type       INT,
  is_active      BOOLEAN       NOT NULL,
  created_at     TIMESTAMP     NOT NULL,
  updated_at     TIMESTAMP     NOT NULL
) COMMENT 'Metadata-driven pipeline configuration';

MERGE INTO ecom_dev.control.pipeline_config AS t
USING (
  SELECT * FROM VALUES
    ('orders',      'ecom_oltp','incremental_file','ingest_date',  'csv','bronze.orders',     'silver.orders',     array('order_id'),                     'updated_at',1,true),
    ('order_items', 'ecom_oltp','incremental_file','ingest_date',  'csv','bronze.order_items','silver.order_items',array('order_id','order_item_id'),     'updated_at',1,true),
    ('payments',    'ecom_oltp','incremental_file','ingest_date',  'csv','bronze.payments',   'silver.payments',   array('order_id','payment_sequential'),'updated_at',1,true),
    ('reviews',     'ecom_oltp','incremental_file','ingest_date',  'csv','bronze.reviews',    'silver.reviews',    array('review_id'),                    'review_ts', 1,true),
    ('customers',   'ecom_crm', 'snapshot',        'snapshot_date','csv','bronze.customers',  'silver.customers',  array('customer_id'),                  'updated_at',2,true),
    ('products',    'ecom_pim', 'snapshot',        'snapshot_date','csv','bronze.products',   'silver.products',   array('product_id'),                   'updated_at',2,true),
    ('sellers',     'ecom_pim', 'snapshot',        'snapshot_date','csv','bronze.sellers',    'silver.sellers',    array('seller_id'),                    'updated_at',1,true)
  AS v(source_table, source_system, load_type, partition_key, file_format, bronze_table, silver_table, primary_keys, order_by_col, scd_type, is_active)
) AS s
ON t.source_table = s.source_table
WHEN MATCHED THEN UPDATE SET
  t.source_system = s.source_system, t.load_type = s.load_type, t.partition_key = s.partition_key,
  t.file_format = s.file_format, t.bronze_table = s.bronze_table, t.silver_table = s.silver_table,
  t.primary_keys = s.primary_keys, t.order_by_col = s.order_by_col, t.scd_type = s.scd_type,
  t.is_active = s.is_active, t.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (source_table, source_system, load_type, partition_key, file_format, bronze_table, silver_table,
   primary_keys, order_by_col, scd_type, is_active, created_at, updated_at)
  VALUES (s.source_table, s.source_system, s.load_type, s.partition_key, s.file_format, s.bronze_table, s.silver_table,
   s.primary_keys, s.order_by_col, s.scd_type, s.is_active, current_timestamp(), current_timestamp());

   SELECT source_table, load_type, partition_key, primary_keys, scd_type FROM ecom_dev.control.pipeline_config;
DESCRIBE DETAIL ecom_dev.control.pipeline_config;




CREATE TABLE IF NOT EXISTS ecom_dev.control.watermark (
  table_name           STRING    NOT NULL,
  layer                STRING    NOT NULL,
  watermark_column     STRING,
  last_watermark_value STRING    NOT NULL,
  last_run_id          STRING,
  last_success_ts      TIMESTAMP,
  updated_at           TIMESTAMP NOT NULL
) COMMENT 'Incremental load bookmarks per target table';

MERGE INTO ecom_dev.control.watermark AS t
USING (
  SELECT * FROM VALUES
    ('silver.orders'),('silver.order_items'),('silver.payments'),('silver.reviews'),
    ('silver.customers'),('silver.products'),('silver.sellers')
  AS v(table_name)
) AS s
ON t.table_name = s.table_name
WHEN NOT MATCHED THEN INSERT
  (table_name, layer, watermark_column, last_watermark_value, last_run_id, last_success_ts, updated_at)
  VALUES (s.table_name, 'silver', '_ingest_ts', '1900-01-01 00:00:00', NULL, NULL, current_timestamp());



SELECT table_name, layer, watermark_column, last_watermark_value FROM ecom_dev.control.watermark ORDER BY table_name;

CREATE TABLE IF NOT EXISTS ecom_dev.control.pipeline_run_audit (
  run_id         STRING    NOT NULL,
  job_name       STRING,
  layer          STRING    NOT NULL,
  table_name     STRING    NOT NULL,
  status         STRING    NOT NULL,
  start_ts       TIMESTAMP NOT NULL,
  end_ts         TIMESTAMP,
  duration_sec   DOUBLE,
  rows_read      BIGINT,
  rows_written   BIGINT,
  rows_rejected  BIGINT,
  target_version BIGINT,
  error_message  STRING,
  bundle_target  STRING,
  triggered_by   STRING
) COMMENT 'One row per table per layer per run';


INSERT INTO ecom_dev.control.pipeline_run_audit
VALUES ('test-run-001','manual_test','bronze','bronze.orders','SUCCESS',
        current_timestamp(), current_timestamp(), 1.5, 1000, 995, 5, 1, NULL, 'dev', 'manual');

SELECT * FROM ecom_dev.control.pipeline_run_audit;

DELETE FROM ecom_dev.control.pipeline_run_audit WHERE run_id = 'test-run-001';

CREATE TABLE IF NOT EXISTS ecom_dev.control.file_ingestion_log (
  table_name       STRING    NOT NULL,
  file_path        STRING    NOT NULL,
  file_size_bytes  BIGINT,
  file_modified_ts TIMESTAMP,
  content_hash     STRING,
  row_count        BIGINT,
  status           STRING    NOT NULL,
  run_id           STRING,
  ingested_at      TIMESTAMP NOT NULL
) COMMENT 'One row per landed file: traceability and duplicate-file protection';

SHOW TABLES IN ecom_dev.control;

